In [1]:
import numpy as np
import pandas as pd
from glob import glob

from _analysis import load_jsons

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/weissl/miniconda3/envs/hynea/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth


In [2]:
usr_path = "/home/weissl"
mimicry_i_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_class*/*.csv")]
mimicry_i_eff_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_eff_class*/*.csv")]
mimicry_i_vit_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_vit_class*/*.csv")]
mimicry_c_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_b*/*.csv")]

hynea_i_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_1*/*.csv")]
hynea_i_eff_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/hynea_custom_eff/*/*.csv")]
hynea_i_vit_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/hynea_custom_vit/*/*.csv")]
hynea_c_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_b*/*.csv")]
hynea_y_data = [pd.read_csv(f) for f in glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_y*/*.csv")]

In [3]:
mimicry_i = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_class*/*.json"))
mimicry_i_eff = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_eff_class*/*.json"))
mimicry_i_vit = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_vit_class*/*.json"))
mimicry_c = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/mimicry/runs/runs/*_custom_b*/*.json"))

hynea_i = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_1*/*.json"))
hynea_i_eff = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/hynea_custom_eff/*/*.json"))
hynea_i_vit = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/hynea_custom_vit/*/*.json"))
hynea_c = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_b*/*.json"))
hynea_y = load_jsons(glob(f"{usr_path}/Projects/SMOO/defaults/hynea/runs/*/*_custom_y*/*.json"))

mrm_path = f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_"
mrm_i = load_jsons(glob(mrm_path + "/*/*.json"))
mrm_i["runtime"] = mrm_i["runtime_seconds"]

mrm_i_eff = load_jsons(glob(f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_eff/*/*.json"))
mrm_i_eff["runtime"] = mrm_i_eff["runtime_seconds"]

mrm_i_vit = load_jsons(glob(f"{usr_path}/PycharmProjects/genai_tigs/experiments/imagenet_experiments/imagenet_sd/ conditional_sd/imagenet_sd/test_vit/*/*.json"))
mrm_i_vit["runtime"] = mrm_i_vit["runtime_seconds"]

mrm_c = load_jsons(glob(f"{usr_path}/PycharmProjects/genai_tigs/sd_weights/celebahq_generatorlow/*.json"))
mrm_c["runtime"] = mrm_c["runtime_seconds"]

mrm_y = load_jsons(glob(f"{usr_path}/PycharmProjects/genai_tigs/yolo_sd/test_/*/*.json"))
mrm_y["runtime"] = mrm_y["runtime_seconds"]

In [4]:
def get_safe_times(df, thresh: int = 30_000):
    """Sometimes time is glitched when logging -> remove those glitched readings."""
    return df["runtime"][df["runtime"] < thresh]

In [6]:
"""Get stats for mimicry"""
mimicry_i_budget = mimicry_i["budget_used"] + mimicry_i["w0_trials"] + mimicry_i["wn_trials"]
mimicry_i_runtime = get_safe_times(mimicry_i)

mimicry_i_eff_budget = mimicry_i_eff["budget_used"] + mimicry_i_eff["w0_trials"] + mimicry_i_eff["wn_trials"]
mimicry_i_eff_runtime = get_safe_times(mimicry_i_eff)

mimicry_i_vit_budget = mimicry_i_vit["budget_used"] + mimicry_i_vit["w0_trials"] + mimicry_i_vit["wn_trials"]
mimicry_i_vit_runtime = get_safe_times(mimicry_i_vit)

mimicry_c_budget = mimicry_c["budget_used"] + mimicry_c["w0_trials"] + mimicry_c["wn_trials"]
mimicry_c_runtime = get_safe_times(mimicry_c)


"""
Get extrapolate stats for mimicry with diffusion

>%%timeit
>with torch.no_grad():
>   manipulator.get_diff_steps([1]*100)

>%%timeit
>with torch.no_grad():
>    manipulator.get_diff_steps([1]*100)
"""
extrapolate_i = (mimicry_i_budget * 45.9) / 100
extrapolate_i_eff = (mimicry_i_eff_budget * 45.9) / 100
extrapolate_i_vit = (mimicry_i_vit_budget * 45.9) / 100
extrapolate_c = (mimicry_c_budget * 47.2) / 100

print("Budget Used:\n")
print(f"Mimicry ImageNet (WideResNet): {mimicry_i_budget.mean():.2f}, {mimicry_i_budget.std():.2f}")
print(f"Mimicry ImageNet (EfficientNetV2): {mimicry_i_eff_budget.mean():.2f}, {mimicry_i_eff_budget.std():.2f}")
print(f"Mimicry ImageNet (ViT): {mimicry_i_vit_budget.mean():.2f}, {mimicry_i_vit_budget.std():.2f}")
print(f"Mimicry CelebA: {mimicry_c_budget.mean():.2f}, {mimicry_c_budget.std():.2f}\n")

print("Runtime: \n")
print(f"Mimicry ImageNet (WideResNet): {mimicry_i_runtime.mean():.2f}, {mimicry_i_runtime.std():.2f}")
print(f"Mimicry ImageNet (EfficientNetV2): {mimicry_i_eff_runtime.mean():.2f}, {mimicry_i_eff_runtime.std():.2f}")
print(f"Mimicry ImageNet (ViT): {mimicry_i_vit_runtime.mean():.2f}, {mimicry_i_vit_runtime.std():.2f}")
print(f"Mimicry CelebA: {mimicry_c_runtime.mean():.2f}, {mimicry_c_runtime.std():.2f}\n")

ex_i = np.concatenate([extrapolate_i, extrapolate_i_eff, extrapolate_i_vit])
print(f"Extrapolate ImageNet: {ex_i.mean():1.0f}")
print(f"Extrapolate CelebA: {extrapolate_c.mean():1.0f}\n")

Budget Used:

Mimicry ImageNet (WideResNet): 2498.16, 390.57
Mimicry ImageNet (EfficientNetV2): 2485.19, 370.40
Mimicry ImageNet (ViT): 2529.10, 291.56
Mimicry CelebA: 2672.25, 38.12

Runtime: 

Mimicry ImageNet (WideResNet): 108.53, 16.93
Mimicry ImageNet (EfficientNetV2): 110.01, 16.12
Mimicry ImageNet (ViT): 120.80, 14.02
Mimicry CelebA: 51.79, 3.53

Extrapolate ImageNet: 1149
Extrapolate CelebA: 1261



In [7]:
"""Get stats for hynea"""
hynea_i_budget = hynea_i["budget_used"]
hynea_i_runtime = get_safe_times(hynea_i)

hynea_i_eff_budget = hynea_i_eff["budget_used"]
hynea_i_eff_runtime = get_safe_times(hynea_i_eff)

hynea_i_vit_budget = hynea_i_vit["budget_used"]
hynea_i_vit_runtime = get_safe_times(hynea_i_vit)

hynea_c_budget = hynea_c["budget_used"]
hynea_c_runtime = get_safe_times(hynea_c)

hynea_y_budget = hynea_y["budget_used"]
hynea_y_runtime = get_safe_times(hynea_y)

print("Budget Used:\n")
print(f"HyNeA ImageNet (WideResNet): {hynea_i_budget.mean():.2f}, {hynea_i_budget.std():.2f}")
print(f"HyNeA ImageNet (EfficientNetV2): {hynea_i_eff_budget.mean():.2f}, {hynea_i_eff_budget.std():.2f}")
print(f"HyNeA ImageNet (ViT): {hynea_i_vit_budget.mean():.2f}, {hynea_i_vit_budget.std():.2f}")
print(f"HyNeA CelebA: {hynea_c_budget.mean():.2f}, {hynea_c_budget.std():.2f}")
print(f"HyNeA Guericke: {hynea_y_budget.mean():.2f}, {hynea_y_budget.std():.2f}\n")

print("Runtime: \n")
print(f"HyNeA ImageNet (WideResNet): {hynea_i_runtime.mean():.2f}, {hynea_i_runtime.std():.2f}")
print(f"HyNeA ImageNet (EfficientNetV2): {hynea_i_eff_runtime.mean():.2f}, {hynea_i_eff_runtime.std():.2f}")
print(f"HyNeA ImageNet (ViT): {hynea_i_vit_runtime.mean():.2f}, {hynea_i_vit_runtime.std():.2f}")
print(f"HyNeA CelebA: {hynea_c_runtime.mean():.2f}, {hynea_c_runtime.std():.2f}")
print(f"HyNeA Guericke: {hynea_y_runtime.mean():.2f}, {hynea_y_runtime.std():.2f}\n")

Budget Used:

HyNeA ImageNet (WideResNet): 25.29, 26.72
HyNeA ImageNet (EfficientNetV2): 25.48, 28.30
HyNeA ImageNet (ViT): 16.55, 17.05
HyNeA CelebA: 30.30, 31.43
HyNeA Guericke: 5.71, 5.69

Runtime: 

HyNeA ImageNet (WideResNet): 94.41, 99.26
HyNeA ImageNet (EfficientNetV2): 98.38, 109.45
HyNeA ImageNet (ViT): 64.07, 66.16
HyNeA CelebA: 220.89, 228.82
HyNeA Guericke: 113.03, 111.70



In [7]:
"""Get stats for GIFTbench"""
mrm_i_runtime = get_safe_times(mrm_i)
mrm_i_budget = mrm_i["budget"]

mrm_i_eff_runtime = get_safe_times(mrm_i_eff)
mrm_i_eff_budget = mrm_i_eff["budget"]

mrm_i_vit_runtime = get_safe_times(mrm_i_vit)
mrm_i_vit_budget = mrm_i_vit["budget"]

mrm_c_runtime = get_safe_times(mrm_c)
mrm_c_budget = mrm_c["budget"]

mrm_y_runtime = get_safe_times(mrm_y)
mrm_y_budget = mrm_y["budget"]

print("Budget Used:\n")
print(f"GIFTBench ImageNet (WideResNet): {mrm_i_budget.mean():.2f}, {mrm_i_budget.std():.2f}")
print(f"GIFTBench ImageNet (EfficientNetV2): {mrm_i_eff_budget.mean():.2f}, {mrm_i_eff_budget.std():.2f}")
print(f"GIFTBench ImageNet (ViT): {mrm_i_vit_budget.mean():.2f}, {mrm_i_vit_budget.std():.2f}")
print(f"GIFTBench CelebA: {mrm_c_budget.mean():.2f}, {mrm_c_budget.std():.2f}\n")
print(f"GIFTBench Guericke: {mrm_y_budget.mean():.2f}, {mrm_y_budget.std():.2f}")

print("Runtime: \n")
print(f"GIFTBench ImageNet (WideResNet): {mrm_i_runtime.mean():.2f}, {mrm_i_runtime.std():.2f}")
print(f"GIFTBench ImageNet (EfficientNetV2): {mrm_i_eff_runtime.mean():.2f}, {mrm_i_eff_runtime.std():.2f}")
print(f"GIFTBench ImageNet (ViT): {mrm_i_vit_runtime.mean():.2f}, {mrm_i_vit_runtime.std():.2f}")
print(f"GIFTBench CelebA: {mrm_c_runtime.mean():.2f}, {mrm_c_runtime.std():.2f}")
print(f"GIFTBench Guericke: {mrm_y_runtime.mean():.2f}, {mrm_y_runtime.std():.2f}")

Budget Used:

GIFTBench ImageNet (WideResNet): 699.63, 748.59
GIFTBench ImageNet (EfficientNetV2): 282.00, 221.05
GIFTBench ImageNet (ViT): 318.25, 267.82
GIFTBench CelebA: 1796.00, 1088.11

GIFTBench Guericke: 232.12, 245.72
Runtime: 

GIFTBench ImageNet (WideResNet): 205.74, 198.99
GIFTBench ImageNet (EfficientNetV2): 190.39, 149.50
GIFTBench ImageNet (ViT): 212.70, 179.63
GIFTBench CelebA: 1217.24, 735.71
GIFTBench Guericke: 174.58, 165.02
